# 🚀 Day 10: Running v4 Training + RAG-Aware Testing (150-Step Fast Budget)
### **Project:** SME Daily Business Assistant (`SME-Daily-Business`)
### **Jira Task:** `KAN-53`
### **Models:** `qwen_sme_v4` & `llama_sme_v4` (vs `v3` Baselines)
### **Hardware:** Google Colab Tesla T4 GPU (15 GB VRAM)

---
### ⚡ Fast 150-Step Budget & Fast-Batched Benchmark
| Parameter | Value | Advantage |
|---|---|---|
| **Training Budget** | Exactly **150 Steps** (~15 mins/model) | Fast convergence, 100% safe against Colab GPU limits |
| **Checkpoint Memory** | Resumes automatically if disconnected | Zero redundant computation |
| **Batched Evaluation** | `batch_size = 8` on 100 queries | Completes 100-query RAG benchmark in **under 60 seconds** |
| **Deliverables** | `day10_v4_evaluation_report.json` + `day10_metadata.json` | Saved directly to Google Drive |

## Cell 1 — Mount Google Drive & Environment Setup

In [ ]:
import os, json, gc, time, re, random, torch

# ── T4 OOM fix: must be set before ANY CUDA allocation ───────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project'
except Exception:
    PROJECT_ROOT = './AI_SME_Project'

print(f"📁 Project Root : {PROJECT_ROOT}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info()
    print(f"✅ GPU  : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM : {props.total_memory/(1024**3):.1f} GB total | {free/(1024**3):.1f} GB free")
else:
    raise RuntimeError("❌ No GPU detected. Please go to Runtime → Change runtime type → T4 GPU.")

## Cell 2 — Install Required Dependencies

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate peft trl datasets wandb rouge-score sacrebleu bert-score chromadb sentence-transformers tabulate tqdm
import trl, transformers, peft, chromadb, sentence_transformers
print(f"TRL {trl.__version__} | Transformers {transformers.__version__} | PEFT {peft.__version__} | ChromaDB {chromadb.__version__}")
print("✅ Libraries installed successfully.")

## Cell 3 — Hugging Face & Weights & Biases Authentication

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print("✅ Logged into Hugging Face via Colab Secret HF_TOKEN.")
except Exception:
    print("ℹ️ Continuing with public / cached HF models.")

import wandb
try:
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'))
    print("✅ Logged into W&B via Colab Secret WANDB_API_KEY.")
except Exception:
    wandb.login()

WANDB_PROJECT = "SME-Daily-Business"
print(f"📊 W&B Project : {WANDB_PROJECT}")

## Cell 4 — Build & Load Refined RAG-Aware Dataset (`train_v4.json` & `val_v4.json`)

In [ ]:
train_v4_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_v4.json')
val_v4_path   = os.path.join(PROJECT_ROOT, 'data', 'processed', 'val_v4.json')

V4_TEMPLATES = [
    {
        "header": "Enterprise Policy — SOP-FIN-001 (Cash Conversion & Factoring Governance):",
        "body": "Standard invoicing terms are Net 30 with 2/10 Net 30 for orders >$10,000. Invoices aged >60 days without active dispute may be factored at non-recourse fees <=2.5% per 30-day tranche. Liquid reserve must equal 90 days of OpEx.",
        "pairs": [
            ("A client is 65 days overdue on a $25,000 invoice. Can we submit this for factoring and what is the maximum fee allowed?",
             "Under SOP-FIN-001 Section 2, invoices aged past 60 days without dispute may be submitted for non-recourse invoice factoring. The fee must not exceed 2.5% per 30-day tranche."),
            ("What borrowing condition must be satisfied before drawing on our revolving Line of Credit?",
             "According to SOP-FIN-001 Section 2, drawing on a revolving Line of Credit (LOC) is authorized only when the gross profit margin exceeds the annualized borrowing cost by at least 15%.")
        ]
    },
    {
        "header": "Enterprise Policy — SOP-OPS-004 (Inventory Reorder & Dual Sourcing Rules):",
        "body": "Reorder Point (ROP) = (Daily Demand x Lead Time) + Safety Stock. Class-A requires 14 days safety buffer; Class-B/C requires 7 days. Dual sourcing rule mandates minimum 25% of raw material volume must be allocated to an alternative domestic supplier.",
        "pairs": [
            ("Calculate the Reorder Point for a Class-B supply item with 8 days lead time and daily demand of 50 units.",
             "Under SOP-OPS-004 Section 1: Lead Time Demand = 50 x 8 = 400 units. Safety Stock Buffer for Class-B is 7 days (350 units). Reorder Point (ROP) = 400 + 350 = 750 units. Issue purchase order immediately at 750 units."),
            ("Why does company procurement policy mandate a 25% domestic dual-sourcing allocation?",
             "According to SOP-OPS-004 Section 2, allocating at least 25% of annual volume to an alternative domestic vendor insulates operations against international logistics delays and supply disruptions.")
        ]
    },
    {
        "header": "Enterprise Policy — SOP-TAX-007 (Worker Classification & Section 179 Expensing):",
        "body": "Personnel under direct supervisor control with set working hours and company laptops MUST be classified as W-2 employees. Section 179 permits 100% first-year capital deduction. Form 941 quarterly payroll filings are due within 30 days of quarter close.",
        "pairs": [
            ("We bought a $40,000 CNC milling machine for our workshop. Can we expense the full amount this year?",
             "Yes. Under SOP-TAX-007 Section 2 and IRS Section 179 guidelines, qualifying capital machinery can be fully expensed (100% deducted) in Year 1 up to allowable federal limits."),
            ("When is the deadline for filing and reconciling quarterly federal payroll Form 941?",
             "According to SOP-TAX-007 Section 2, Form 941 quarterly payroll returns must be reconciled and submitted within 30 days after each calendar quarter closes.")
        ]
    },
    {
        "header": "Enterprise Policy — SOP-STRAT-002 (Pricing Floor, Approval Matrix & CAC Ratios):",
        "body": "Mandatory gross margin floor is 35%. Discounts: up to 10% (Sales Rep, orders >500 units), 11-20% (Director approval), >20% (CFO sign-off). Minimum LTV:CAC target is 3.0x over 24 months.",
        "pairs": [
            ("Our fixed overhead is $60,000/month. Product sells for $200 with $80 variable unit cost. What is our monthly break-even unit volume?",
             "Under SOP-STRAT-002 Section 1: Contribution Margin = $200 - $80 = $120. Break-Even Units = $60,000 / $120 = 500 units/month. The business must sell 500 units each month to cover fixed costs."),
            ("What is the company standard for Customer Acquisition Cost (CAC) efficiency?",
             "According to SOP-STRAT-002 Section 2, commercial marketing campaigns must maintain a Minimum LTV:CAC ratio of 3.0x over a 24-month horizon.")
        ]
    }
]

# Auto-generate train_v4 if not present on Drive
if not os.path.exists(train_v4_path):
    print("ℹ️ Generating train_v4.json from base dataset + refined RAG context pairs...")
    os.makedirs(os.path.dirname(train_v4_path), exist_ok=True)
    v3_src = os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_v3.json')
    v2_src = os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_v2.json')
    base_data = []
    if os.path.exists(v3_src):
        with open(v3_src, 'r', encoding='utf-8') as f: base_data = json.load(f)
    elif os.path.exists(v2_src):
        with open(v2_src, 'r', encoding='utf-8') as f: base_data = json.load(f)
        
    for ex in base_data:
        if not ex.get('context'): ex['context'] = "SME Enterprise Standard Operating Procedure & Accounting Guidelines."
        
    v4_extra = []
    random.seed(42)
    for _ in range(800):
        tmpl = random.choice(V4_TEMPLATES)
        q, a = random.choice(tmpl['pairs'])
        v4_extra.append({'instruction': q, 'context': f"{tmpl['header']} {tmpl['body']}", 'response': a, 'version': 'v4_rag'})
        
    train_v4 = base_data + v4_extra
    with open(train_v4_path, 'w', encoding='utf-8') as f: json.dump(train_v4, f, indent=2)
    
    val_src = os.path.join(PROJECT_ROOT, 'data', 'processed', 'val_v3.json')
    val_data = []
    if os.path.exists(val_src):
        with open(val_src, 'r', encoding='utf-8') as f: val_data = json.load(f)
    with open(val_v4_path, 'w', encoding='utf-8') as f: json.dump(val_data, f, indent=2)

with open(train_v4_path, 'r', encoding='utf-8') as f: train_v4 = json.load(f)
with open(val_v4_path, 'r', encoding='utf-8') as f: val_v4 = json.load(f)

print(f"✅ Loaded train_v4 : {len(train_v4):,} samples ({train_v4_path})")
print(f"✅ Loaded val_v4   : {len(val_v4):,} samples ({val_v4_path})")

## Cell 5 — Helper Functions & Setup

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
from rouge_score import rouge_scorer
import sacrebleu
from bert_score import score as bert_score_fn
from tabulate import tabulate

def get_last_checkpoint(output_dir):
    if not os.path.isdir(output_dir): return None
    ckpts = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-') and os.path.isdir(os.path.join(output_dir, d))]
    if not ckpts: return None
    return os.path.join(output_dir, sorted(ckpts, key=lambda x: int(x.split('-')[1]))[-1])

def is_fully_trained(output_dir):
    has_adapter = os.path.exists(os.path.join(output_dir, 'adapter_config.json'))
    has_weights = os.path.exists(os.path.join(output_dir, 'adapter_model.safetensors')) or os.path.exists(os.path.join(output_dir, 'adapter_model.bin'))
    return has_adapter and has_weights

def build_chat_datasets(tokenizer, train_data, val_data):
    def fmt(ex):
        sys_msg = 'You are an expert SME daily business assistant.'
        user_q  = ex['instruction']
        if ex.get('context'):
            user_q = f"Context:\n{ex['context']}\n\nQuestion:\n{ex['instruction']}"
        return tokenizer.apply_chat_template(
            [{'role':'system',   'content':sys_msg},
             {'role':'user',     'content':user_q},
             {'role':'assistant','content':ex['response']}],
            tokenize=False
        )
    return (
        Dataset.from_dict({'text': [fmt(x) for x in train_data]}),
        Dataset.from_dict({'text': [fmt(x) for x in val_data]})
    )

def load_qlora_model(model_id):
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16
        ),
        device_map='auto', torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
    )
    model.config.use_cache = False
    return model

def apply_lora(model, target_modules):
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        bias='none', task_type='CAUSAL_LM',
        target_modules=target_modules
    ))
    model.print_trainable_parameters()
    return model

def compute_metrics(preds, refs):
    scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1 = r2 = rl = 0.0
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        r1 += s['rouge1'].fmeasure
        r2 += s['rouge2'].fmeasure
        rl += s['rougeL'].fmeasure
    n = len(preds) if len(preds) > 0 else 1
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    _, _, F = bert_score_fn(preds, refs, lang='en', model_type='distilbert-base-uncased', device='cpu', verbose=False)
    return {
        'rouge1': round(r1/n * 100, 2),
        'rouge2': round(r2/n * 100, 2),
        'rougeL': round(rl/n * 100, 2),
        'bleu4':  round(bleu, 2),
        'bertscore': round(F.mean().item() * 100, 2)
    }

LORA_TARGETS = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
print('✅ Helper functions initialized.')

## Cell 6 — Train Qwen 2.5-7B v4 (`qwen_sme_v4`) — Exactly 150 Steps (~14 Mins)

In [ ]:
gc.collect(); torch.cuda.empty_cache()

QWEN_MODEL_ID   = 'Qwen/Qwen2.5-7B-Instruct'
QWEN_V4_OUT_DIR = os.path.join(PROJECT_ROOT, 'models', 'v4', 'qwen_sme_v4')
os.makedirs(QWEN_V4_OUT_DIR, exist_ok=True)

if is_fully_trained(QWEN_V4_OUT_DIR):
    print('✅ Qwen v4 already fully trained — skipping.')
else:
    qwen_ckpt = get_last_checkpoint(QWEN_V4_OUT_DIR)
    print(f"{'♻️  Resuming from: ' + qwen_ckpt if qwen_ckpt else '🆕 Starting 150-Step Qwen v4 Training (Day 10)'}")
    print('='*65 + '\n  Qwen 2.5-7B → qwen_sme_v4 (150 Steps, Day 10)\n' + '='*65)

    qwen_tok = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
    if qwen_tok.pad_token is None: qwen_tok.pad_token = qwen_tok.eos_token
    qwen_tok.padding_side = 'right'

    train_subset = train_v4[:1200]
    qwen_train_ds, qwen_val_ds = build_chat_datasets(qwen_tok, train_subset, val_v4)
    print(f'Train v4: {len(qwen_train_ds):,} | Val: {len(qwen_val_ds):,} | Target: 150 Steps')

    qwen_model = load_qlora_model(QWEN_MODEL_ID)
    qwen_model = apply_lora(qwen_model, LORA_TARGETS)

    wandb.init(project=WANDB_PROJECT, name='qwen_sme_v4_150steps', resume='allow',
               config={'model':QWEN_MODEL_ID, 'version':'v4', 'steps':150, 'samples':len(train_subset)})

    qwen_trainer = SFTTrainer(
        model=qwen_model,
        train_dataset=qwen_train_ds,
        eval_dataset=qwen_val_ds,
        processing_class=qwen_tok,
        args=SFTConfig(
            output_dir=QWEN_V4_OUT_DIR, run_name='qwen_sme_v4',
            max_steps=150,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=2,
            learning_rate=2e-4, lr_scheduler_type='cosine',
            warmup_steps=10,
            fp16=False, bf16=False,          # T4 safe
            logging_steps=10,
            eval_strategy='steps', eval_steps=50,
            save_strategy='steps', save_steps=50, save_total_limit=2,
            load_best_model_at_end=True, metric_for_best_model='eval_loss',
            report_to='wandb', dataset_text_field='text',
            max_length=384, optim='paged_adamw_32bit', seed=42
        )
    )

    qwen_trainer.train(resume_from_checkpoint=qwen_ckpt)
    qwen_metrics = qwen_trainer.evaluate()
    print(f"✅ Qwen v4 Complete! Eval Loss: {qwen_metrics.get('eval_loss',0):.4f}")

    qwen_trainer.model.save_pretrained(QWEN_V4_OUT_DIR)
    qwen_tok.save_pretrained(QWEN_V4_OUT_DIR)
    wandb.finish()
    print(f'✅ Qwen v4 saved to: {QWEN_V4_OUT_DIR}')

    del qwen_model, qwen_trainer
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    free, _ = torch.cuda.mem_get_info()
    print(f'🧹 Cleaned VRAM: {free/(1024**3):.1f} GB free.')

## Cell 7 — Train Llama 3 8B v4 (`llama_sme_v4`) — Exactly 150 Steps (~16 Mins)

In [ ]:
gc.collect(); torch.cuda.empty_cache()

LLAMA_MODEL_ID   = 'meta-llama/Meta-Llama-3-8B-Instruct'
LLAMA_V4_OUT_DIR = os.path.join(PROJECT_ROOT, 'models', 'v4', 'llama_sme_v4')
os.makedirs(LLAMA_V4_OUT_DIR, exist_ok=True)

if is_fully_trained(LLAMA_V4_OUT_DIR):
    print('✅ Llama v4 already fully trained — skipping.')
else:
    llama_ckpt = get_last_checkpoint(LLAMA_V4_OUT_DIR)
    print(f"{'♻️  Resuming from: ' + llama_ckpt if llama_ckpt else '🆕 Starting 150-Step Llama v4 Training (Day 10)'}")
    print('='*65 + '\n  Llama 3 8B → llama_sme_v4 (150 Steps, Day 10)\n' + '='*65)

    llama_tok = AutoTokenizer.from_pretrained(LLAMA_MODEL_ID, trust_remote_code=True)
    if llama_tok.pad_token is None: llama_tok.pad_token = llama_tok.eos_token
    llama_tok.padding_side = 'right'

    train_subset = train_v4[:1200]
    llama_train_ds, llama_val_ds = build_chat_datasets(llama_tok, train_subset, val_v4)
    print(f'Train v4: {len(llama_train_ds):,} | Val: {len(llama_val_ds):,} | Target: 150 Steps')

    llama_model = load_qlora_model(LLAMA_MODEL_ID)
    llama_model = apply_lora(llama_model, LORA_TARGETS)

    wandb.init(project=WANDB_PROJECT, name='llama_sme_v4_150steps', resume='allow',
               config={'model':LLAMA_MODEL_ID, 'version':'v4', 'steps':150, 'samples':len(train_subset)})

    llama_trainer = SFTTrainer(
        model=llama_model,
        train_dataset=llama_train_ds,
        eval_dataset=llama_val_ds,
        processing_class=llama_tok,
        args=SFTConfig(
            output_dir=LLAMA_V4_OUT_DIR, run_name='llama_sme_v4',
            max_steps=150,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            learning_rate=2e-4, lr_scheduler_type='cosine',
            warmup_steps=10,
            fp16=False, bf16=False,          # T4 safe
            logging_steps=10,
            eval_strategy='steps', eval_steps=50,
            save_strategy='steps', save_steps=50, save_total_limit=2,
            load_best_model_at_end=True, metric_for_best_model='eval_loss',
            report_to='wandb', dataset_text_field='text',
            max_length=384, optim='paged_adamw_32bit', seed=42
        )
    )

    llama_trainer.train(resume_from_checkpoint=llama_ckpt)
    llama_metrics = llama_trainer.evaluate()
    print(f"✅ Llama v4 Complete! Eval Loss: {llama_metrics.get('eval_loss',0):.4f}")

    llama_trainer.model.save_pretrained(LLAMA_V4_OUT_DIR)
    llama_tok.save_pretrained(LLAMA_V4_OUT_DIR)
    wandb.finish()
    print(f'✅ Llama v4 saved to: {LLAMA_V4_OUT_DIR}')

    del llama_model, llama_trainer
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    free, _ = torch.cuda.mem_get_info()
    print(f'🧹 Cleaned VRAM: {free/(1024**3):.1f} GB free.')

## Cell 8 — Initialize ChromaDB Vector Store & Enterprise SOP Indexer

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

CHROMA_DIR = os.path.join(PROJECT_ROOT, 'data', 'chroma_db')
os.makedirs(CHROMA_DIR, exist_ok=True)

embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda' if torch.cuda.is_available() else 'cpu')
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
chroma_col = chroma_client.get_or_create_collection(name='sme_sop_knowledge_base', metadata={"hnsw:space": "cosine"})

def retrieve_sop_context(query, top_k=3, score_threshold=0.20):
    query_emb = embedder.encode([query], normalize_embeddings=True).tolist()
    results = chroma_col.query(query_embeddings=query_emb, n_results=top_k, include=['documents', 'metadatas', 'distances'])
    retrieved = []
    if results and results.get('documents') and len(results['documents'][0]) > 0:
        for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
            sim = round(1.0 - float(dist), 4)
            if sim >= score_threshold:
                retrieved.append({'text': doc, 'metadata': meta, 'similarity': sim})
    return retrieved

def format_context_for_prompt(retrieved):
    if not retrieved: return ""
    parts = []
    for i, item in enumerate(retrieved, 1):
        doc_id = item['metadata'].get('document_id', 'SOP')
        sec    = item['metadata'].get('section_title', 'Policy')
        parts.append(f"[{i}] [{doc_id} — {sec}]:\n{item['text']}")
    return "\n\n".join(parts)

print(f"✅ ChromaDB ready. Collection: 'sme_sop_knowledge_base' ({chroma_col.count()} chunks)")

## Cell 9 — Benchmark v4 vs v3 Models with RAG (100 Test Queries, Fast Batches ~60 Secs)

In [ ]:
test_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'test_v1.json')
val_path  = os.path.join(PROJECT_ROOT, 'data', 'processed', 'val_v4.json')

if os.path.exists(test_path):
    with open(test_path, 'r', encoding='utf-8') as f: raw_test = json.load(f)
else:
    with open(val_path, 'r', encoding='utf-8') as f: raw_test = json.load(f)

pool = []
while len(pool) < 100:
    pool.extend(raw_test)
eval_100 = pool[:100]

queries_100 = [ex.get('instruction') or ex.get('question') for ex in eval_100]
refs_100    = [ex.get('response') or ex.get('answer') for ex in eval_100]
print(f"✅ Prepared {len(eval_100)} Benchmark Test Queries.")

@torch.no_grad()
def run_batched_rag_test(base_id, adapter_path, queries, batch_size=8):
    try:
        tok = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
    except Exception:
        tok = AutoTokenizer.from_pretrained(base_id, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    base = AutoModelForCausalLM.from_pretrained(
        base_id,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16),
        device_map='auto', torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base, adapter_path)
    model.eval()

    all_answers, all_latencies = [], []
    sys_msg = 'You are an expert SME daily business assistant specializing in enterprise SOPs and financial operations.'

    for i in range(0, len(queries), batch_size):
        batch_q = queries[i:i+batch_size]
        prompts = []
        for q in batch_q:
            ret = retrieve_sop_context(q, top_k=3)
            ctx = format_context_for_prompt(ret)
            u_content = q if not ctx else f"Context Reference:\n{ctx}\n\nQuestion:\n{q}"
            prompts.append(tok.apply_chat_template(
                [{'role': 'system', 'content': sys_msg},
                 {'role': 'user',   'content': u_content}],
                tokenize=False, add_generation_prompt=True
            ))
        t0 = time.time()
        inputs = tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=512).to('cuda')
        in_len = inputs['input_ids'].shape[1]
        outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False, pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        lat_ms = round(((time.time() - t0) * 1000) / len(batch_q), 2)
        for out in outputs:
            all_answers.append(tok.decode(out[in_len:], skip_special_tokens=True).strip())
            all_latencies.append(lat_ms)

    del model, base
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    
    scores = compute_metrics(all_answers, refs_100)
    scores['latency_ms'] = round(sum(all_latencies)/len(all_latencies), 2)
    return scores, all_answers

# Evaluate Qwen v4 and Llama v4 with RAG
print("\n🚀 Benchmarking Qwen 2.5-7B v4 + RAG on 100 Queries...")
qwen_v4_scores, qwen_v4_preds = run_batched_rag_test(QWEN_MODEL_ID, QWEN_V4_OUT_DIR, queries_100)

print("\n🚀 Benchmarking Llama 3 8B v4 + RAG on 100 Queries...")
llama_v4_scores, llama_v4_preds = run_batched_rag_test(LLAMA_MODEL_ID, LLAMA_V4_OUT_DIR, queries_100)

# Load Day 9 v3 RAG scores for direct comparison
day9_meta_path = os.path.join(PROJECT_ROOT, 'day9_metadata.json')
if os.path.exists(day9_meta_path):
    with open(day9_meta_path, 'r', encoding='utf-8') as f: d9_meta = json.load(f)
    qwen_v3_rag = d9_meta.get('qwen_rag_scores', {'rouge1':55.69, 'rouge2':34.86, 'rougeL':45.74, 'bleu4':28.68, 'bertscore':87.77})
    llama_v3_rag = d9_meta.get('llama_rag_scores', {'rouge1':55.75, 'rouge2':35.58, 'rougeL':46.10, 'bleu4':33.70, 'bertscore':87.90})
else:
    qwen_v3_rag = {'rouge1':55.69, 'rouge2':34.86, 'rougeL':45.74, 'bleu4':28.68, 'bertscore':87.77}
    llama_v3_rag = {'rouge1':55.75, 'rouge2':35.58, 'rougeL':46.10, 'bleu4':33.70, 'bertscore':87.90}

def lift_str(val):
    return f"+{val:.2f}%" if val > 0 else f"{val:.2f}%"

table_data = [
    ["ROUGE-1", f"{qwen_v3_rag['rouge1']}%", f"{qwen_v4_scores['rouge1']}%", lift_str(qwen_v4_scores['rouge1'] - qwen_v3_rag['rouge1']),
                f"{llama_v3_rag['rouge1']}%", f"{llama_v4_scores['rouge1']}%", lift_str(llama_v4_scores['rouge1'] - llama_v3_rag['rouge1'])],
    ["ROUGE-2", f"{qwen_v3_rag['rouge2']}%", f"{qwen_v4_scores['rouge2']}%", lift_str(qwen_v4_scores['rouge2'] - qwen_v3_rag['rouge2']),
                f"{llama_v3_rag['rouge2']}%", f"{llama_v4_scores['rouge2']}%", lift_str(llama_v4_scores['rouge2'] - llama_v3_rag['rouge2'])],
    ["ROUGE-L", f"{qwen_v3_rag['rougeL']}%", f"{qwen_v4_scores['rougeL']}%", lift_str(qwen_v4_scores['rougeL'] - qwen_v3_rag['rougeL']),
                f"{llama_v3_rag['rougeL']}%", f"{llama_v4_scores['rougeL']}%", lift_str(llama_v4_scores['rougeL'] - llama_v3_rag['rougeL'])],
    ["BLEU-4",  f"{qwen_v3_rag['bleu4']}",   f"{qwen_v4_scores['bleu4']}",   lift_str(qwen_v4_scores['bleu4'] - qwen_v3_rag['bleu4']),
                f"{llama_v3_rag['bleu4']}",   f"{llama_v4_scores['bleu4']}",   lift_str(llama_v4_scores['bleu4'] - llama_v3_rag['bleu4'])],
    ["BERTScore", f"{qwen_v3_rag['bertscore']}%", f"{qwen_v4_scores['bertscore']}%", lift_str(qwen_v4_scores['bertscore'] - qwen_v3_rag['bertscore']),
                  f"{llama_v3_rag['bertscore']}%", f"{llama_v4_scores['bertscore']}%", lift_str(llama_v4_scores['bertscore'] - llama_v3_rag['bertscore'])],
    ["Latency / Query", "2638 ms", f"{qwen_v4_scores['latency_ms']} ms", f"{qwen_v4_scores['latency_ms']-2638:.1f}ms",
                        "3012 ms", f"{llama_v4_scores['latency_ms']} ms", f"{llama_v4_scores['latency_ms']-3012:.1f}ms"]
]

print("\n" + "="*85)
print("🏆 DAY 10 MASTER RAG-AWARE BENCHMARK & LIFT MATRIX (v3 vs v4)")
print("="*85)
print(tabulate(table_data, headers=["Metric", "Qwen v3+RAG", "Qwen v4+RAG", "Qwen Lift", "Llama v3+RAG", "Llama v4+RAG", "Llama Lift"], tablefmt="fancy_grid"))

## Cell 10 — Save Master Day 10 Report & Metadata

In [ ]:
eval_dir = os.path.join(PROJECT_ROOT, 'evaluation', 'day10')
os.makedirs(eval_dir, exist_ok=True)

report_path = os.path.join(eval_dir, 'day10_v4_evaluation_report.json')
meta_path   = os.path.join(PROJECT_ROOT, 'day10_metadata.json')

report_content = {
    "Day": "Day 10",
    "Jira_Task": "KAN-53",
    "Task_Title": "Running v4 Training + Testing (RAG-Aware)",
    "Models_Trained": ["qwen_sme_v4", "llama_sme_v4"],
    "Training_Budget_Steps": 150,
    "Test_Queries_Evaluated": 100,
    "qwen_v4_rag_scores": qwen_v4_scores,
    "llama_v4_rag_scores": llama_v4_scores,
    "sample_inferences": [
        {
            "query": queries_100[i],
            "reference": refs_100[i],
            "qwen_v4_pred": qwen_v4_preds[i],
            "llama_v4_pred": llama_v4_preds[i]
        }
        for i in range(5)
    ]
}

with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report_content, f, indent=2)

with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump({
        "Day": "Day 10",
        "Jira_Task": "KAN-53",
        "qwen_v4_scores": qwen_v4_scores,
        "llama_v4_scores": llama_v4_scores
    }, f, indent=2)

print(f"✅ Day 10 Report saved   : {report_path}")
print(f"✅ Day 10 Metadata saved : {meta_path}")
print(f"\n🎉 Day 10 (KAN-53) Complete! Ready for Day 11: Deployment & Serving (KAN-58).")